# MEM — dual-memory walkthrough

Manual exploration of the MEM (Multi-scale Embodied Memory) store. We simulate a 15-minute kitchen task: many short subtasks, each producing a video clip and a language summary. We then ask the long-term memory questions like *'did I already add salt?'* and verify it retrieves the right subtask.

Paper: `papers/2026-03-03_mem_vla-long-short-term-memory.pdf`

In [1]:
import numpy as np
import matplotlib.pyplot as plt

from pi_stack.memory.mem import MEM, MEMConfig

rng = np.random.default_rng(0)

def fake_frame(label):
    """Tiny synthetic frame whose content encodes a label (for human inspection)."""
    img = rng.integers(0, 32, size=(8, 8, 3), dtype=np.uint8)
    img[0, 0, 0] = label   # tag
    return img

## 1. Short-term ring buffer

Push 60 frames at 20 Hz over 3 seconds. Configured to keep 1 second at 4 fps → expect 4 frames retained.

In [2]:
mem = MEM(MEMConfig(short_term_video_seconds=1.0, short_term_fps=4))
for i in range(60):
    t = i * 0.05    # 20 Hz incoming camera
    accepted = mem.add_frame(fake_frame(label=i), t=t)
    # only every 5th frame should land (20 Hz -> 4 Hz downsample)
stm = mem.recent_frames()
ts  = mem.short_term.timestamps()
print(f'frames in buffer : {stm.shape[0]} (cap={mem.short_term.max_frames})')
print(f'timestamp range  : {ts.min():.2f}s .. {ts.max():.2f}s')
print(f'frame labels     : {[int(f[0,0,0]) for f in stm]}')

frames in buffer : 4 (cap=4)
timestamp range  : 2.00s .. 2.75s
frame labels     : [40, 45, 50, 55]


## 2. Long-term semantic recall

Simulate a 15-minute kitchen task: 15 subtasks at 1-minute granularity. After everything is logged, ask MEM specific questions and look at top-K hits.

In [3]:
mem = MEM(MEMConfig(long_term_max_summaries=32, embedding_dim=128))

subtasks = [
    'pick up cutting board', 'place cutting board on counter', 'open fridge',
    'pick up lettuce', 'place lettuce on cutting board', 'close fridge',
    'pick up knife', 'chop lettuce', 'transfer lettuce to bowl',
    'open salt shaker', 'add salt to bowl', 'close salt shaker',
    'open olive oil bottle', 'pour olive oil into bowl', 'mix salad with tongs',
]
for i, st in enumerate(subtasks):
    frames = [fake_frame(label=i) for _ in range(8)]
    mem.add_subtask_summary(st, frames=frames, t_start=60.0*i, t_end=60.0*(i+1))

print(f'subtasks logged  : {len(mem.long_term)}')

subtasks logged  : 15


In [4]:
queries = [
    'did I add salt yet?',
    'where did I put the lettuce?',
    'have I closed the fridge?',
    'is the olive oil already poured?',
    'when did I pick up the knife?',
]
for q in queries:
    hits = mem.recall(q, k=3)
    print(f'\nQ: {q}')
    for h in hits:
        print(f'  → t={h.t_start:5.0f}s  subtask={h.subtask!r}')
        print(f'      summary: {h.summary}')


Q: did I add salt yet?
  → t=  600s  subtask='add salt to bowl'
      summary: completed subtask 'add salt to bowl' over 8 frames
  → t=  540s  subtask='open salt shaker'
      summary: completed subtask 'open salt shaker' over 8 frames
  → t=  660s  subtask='close salt shaker'
      summary: completed subtask 'close salt shaker' over 8 frames

Q: where did I put the lettuce?
  → t=  420s  subtask='chop lettuce'
      summary: completed subtask 'chop lettuce' over 8 frames
  → t=  180s  subtask='pick up lettuce'
      summary: completed subtask 'pick up lettuce' over 8 frames
  → t=  480s  subtask='transfer lettuce to bowl'
      summary: completed subtask 'transfer lettuce to bowl' over 8 frames

Q: have I closed the fridge?
  → t=  120s  subtask='open fridge'
      summary: completed subtask 'open fridge' over 8 frames
  → t=  300s  subtask='close fridge'
      summary: completed subtask 'close fridge' over 8 frames
  → t=    0s  subtask='pick up cutting board'
      summary: comple

## 3. Limitations of the default embedder

The default embedder is a bag-of-words hash — fast, deterministic, no deps. But it can't capture synonyms (e.g., 'fridge' vs 'refrigerator'). Swap in a real sentence embedder for production:

```python
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')
mem = MEM(
    config=MEMConfig(embedding_dim=384),
    embedder=lambda text: model.encode(text).astype('float32'),
)
```

## 4. FIFO eviction stress test

Push more subtasks than `long_term_max_summaries`. Oldest entries should drop.

In [5]:
mem = MEM(MEMConfig(long_term_max_summaries=10))
for i in range(25):
    mem.add_subtask_summary(f'subtask_{i}', frames=[fake_frame(label=i)], t_start=i, t_end=i+1)
kept = [e.subtask for e in mem.long_term.entries()]
print('summaries kept    :', kept)
print('# in memory       :', len(mem.long_term))
print('oldest t_start    :', mem.long_term.entries()[0].t_start)

summaries kept    : ['subtask_15', 'subtask_16', 'subtask_17', 'subtask_18', 'subtask_19', 'subtask_20', 'subtask_21', 'subtask_22', 'subtask_23', 'subtask_24']
# in memory       : 10
oldest t_start    : 15.0


## Takeaways

- Short-term: stays fps-bounded; old frames evict cleanly.
- Long-term: cosine recall on bag-of-words embeddings already routes simple keyword queries to the right subtask.
- For the π₀.₇ integration, swap `embedder=` for a real sentence-transformer and `summarizer=` for an Anthropic / HF model.